# Cryptocurrency Volatility Prediction

This notebook walks through the complete pipeline end-to-end — data loading, preprocessing, feature engineering, EDA, model training, and evaluation. Each section corresponds to a source file in `src/` that can also be run as a standalone script.

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
sns.set_theme(style='whitegrid')
print('Setup complete')

## 2. Data Loading & Cleaning

In [ ]:
from src.data_loader import load_raw_data, inspect_data, clean_data, save_clean_data

df_raw = load_raw_data('../data/crypto_historical_prices.csv')
inspect_data(df_raw)

In [ ]:
df_clean = clean_data(df_raw)
save_clean_data(df_clean, '../data/crypto_cleaned.csv')
df_clean.head()

## 3. Feature Engineering

In [ ]:
from src.feature_engineering import compute_features, save_features

df_feat = compute_features(df_clean)
save_features(df_feat, '../data/crypto_features.csv')
print(df_feat.columns.tolist())
df_feat[['date', 'symbol', 'close', 'volatility', 'bb_bandwidth', 'atr', 'liquidity_ratio']].head(10)

## 4. Exploratory Data Analysis

In [ ]:
# Dataset statistics
print(f"Date range : {df_feat['date'].min().date()} to {df_feat['date'].max().date()}")
print(f"Symbols    : {df_feat['symbol'].nunique()}")
print(f"Total rows : {len(df_feat):,}")
df_feat[['close', 'volume', 'volatility', 'log_return']].describe().round(4)

In [ ]:
# Closing price trend — top 6 by market cap
top_syms = df_feat.groupby('symbol')['market_cap'].median().nlargest(6).index

fig, ax = plt.subplots(figsize=(13, 5))
for sym in top_syms:
    s = df_feat[df_feat['symbol'] == sym]
    ax.plot(s['date'], s['close'], label=sym, linewidth=1.4)
ax.set(title='Closing Price Over Time', xlabel='Date', ylabel='Close (USD)')
ax.legend(ncol=3)
plt.tight_layout()
plt.show()

In [ ]:
# Rolling volatility
fig, ax = plt.subplots(figsize=(13, 5))
for sym in top_syms:
    s = df_feat[df_feat['symbol'] == sym]
    ax.plot(s['date'], s['volatility'], label=sym, linewidth=1.2, alpha=0.8)
ax.set(title='Annualised Rolling Volatility', xlabel='Date', ylabel='Volatility')
ax.legend(ncol=3)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
cols = ['volatility', 'log_return', 'volume_change', 'bb_bandwidth',
        'atr', 'liquidity_ratio', 'close_to_ma7', 'volatility_lag_1']
corr = df_feat[cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Log return distribution
from scipy.stats import norm
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df_feat['log_return'].dropna(), bins=200, color='steelblue', density=True, alpha=0.8)
mu, std = df_feat['log_return'].mean(), df_feat['log_return'].std()
x = np.linspace(-0.3, 0.3, 300)
ax.plot(x, norm.pdf(x, mu, std), 'r-', linewidth=2, label='Normal fit')
ax.set(title='Daily Log Return Distribution', xlabel='Log Return', ylabel='Density')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Model Training

In [ ]:
from src.model_training import load_and_prepare, train_random_forest, train_xgboost

X_train, X_test, y_train, y_test, scaler = load_and_prepare('../data/crypto_features.csv')

In [ ]:
rf_metrics  = train_random_forest(X_train, y_train, X_test, y_test)
xgb_metrics = train_xgboost(X_train, y_train, X_test, y_test)

In [ ]:
# Train LSTM (optional — takes longer)
from src.model_training import train_lstm
lstm_metrics = train_lstm(X_train, y_train, X_test, y_test)

In [ ]:
results = pd.DataFrame([rf_metrics, xgb_metrics, lstm_metrics])
print(results[['model', 'RMSE', 'MAE', 'R2']].to_string(index=False))

## 6. Model Evaluation

In [ ]:
import joblib
import xgboost as xgb_lib
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

xgb_model = xgb_lib.XGBRegressor()
xgb_model.load_model('../models/xgboost.json')
y_pred = xgb_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)
print(f'XGBoost  RMSE={rmse:.5f}  MAE={mae:.5f}  R²={r2:.4f}')

In [ ]:
# Actual vs Predicted
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_test, y_pred, alpha=0.3, s=8, color='steelblue')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1.5)
axes[0].set(xlabel='Actual', ylabel='Predicted', title='Actual vs Predicted Volatility')

n = 500
axes[1].plot(y_test[:n], label='Actual',    linewidth=1.2)
axes[1].plot(y_pred[:n], label='Predicted', linewidth=1, linestyle='--', alpha=0.8)
axes[1].set(xlabel='Index', ylabel='Volatility', title='First 500 Test Predictions')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance
fi = pd.DataFrame({'Feature': [
    'log_return', 'volume_change', 'bb_bandwidth', 'atr',
    'liquidity_ratio', 'close_to_ma7', 'close_to_ma30',
    'volatility_lag_1', 'volatility_lag_3', 'volatility_lag_7'
], 'Importance': xgb_model.feature_importances_}).sort_values('Importance')

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(fi['Feature'], fi['Importance'], color='mediumseagreen')
ax.set(title='XGBoost Feature Importances', xlabel='Importance')
plt.tight_layout()
plt.show()

## 7. Inference Example

In [ ]:
from src.predict import predict_from_history

# Change symbol and date to any available value in your dataset
pred = predict_from_history(symbol='BTC', date='2023-06-01')
print(f'Predicted volatility: {pred:.6f}')